**Note:** This notebook is designed for **Google Colab**.

If you see the Colab logo <span style='vertical-align:bottom;'><img src='https://colab.research.google.com/img/colab_favicon_256px.png' width='40' alt='Colab logo'></span> in the top-left corner, you're all set! Please **proceed to Section 1**.

If you don't see the logo (e.g., you are on GitHub), please click the button below to open it in the correct environment:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mparrott-at-wiris/aimodelshare/blob/master/notebooks/Etica_en_Joc_Justice_Challenge.ipynb)

# **Advanced Justice & Equity Challenge: Build & Submit Custom Models**

Welcome to the **Advanced Pathway** of the Ethics at Play (Ètica en Joc) Justice Challenge. 

**Who is this for?** 
This notebook is designed for participants with Python experience (e.g., Scikit-Learn, TensorFlow, PyTorch). Instead of using the gamified apps, you will build, train, and submit your own machine learning models directly to the competition leaderboard.

**The Goal:** 
Train a model to predict recidivism risk (the likelihood of re-offending) using the COMPAS dataset, while balancing accuracy and fairness.

## 🚀 **Quick Start Guide**

To participate in the challenge, complete these 5 steps:

1.  **Install Libraries:** Run the setup cell to install `aimodelshare`.
2.  **Get the Data:** Run the data loading cell to retrieve and preprocess the COMPAS dataset.
3.  **Train Your Model:** Use the provided example (Logistic Regression) or write your own custom training code.
4.  **Connect:** Link this notebook to the Justice Challenge Leaderboard.
5.  **Submit:** Send your trained model and predictions to the leaderboard to see your score.

**Ready? Click the ▶ Play Button on the first cell below to get started.**

---
# **Step 1: Installation**

We need to install the `aimodelshare` library to connect to the competition backend.

In [ ]:
# Install the aimodelshare library
print("Installing required libraries...")
!pip install aimodelshare --upgrade -q --no-warn-script-location > /dev/null 2>&1
print("✅ Installation complete!")

---
# **Step 2: Load & Process Data**

We will use the **COMPAS** dataset, which is the standard dataset used for this challenge. We will perform basic preprocessing to encode categorical variables (like turning text into numbers) so they are ready for machine learning.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load the dataset from a public source
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
data = pd.read_csv(url)

# 2. Select features relevant to the challenge
# We will use a subset of features common in fairness analysis
features = ['sex', 'age', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree']
target = 'two_year_recid'

# Create a clean dataframe
df = data[features + [target]].copy()

# 3. Basic Preprocessing
# Drop rows with missing values
df = df.dropna()

# Map text categories to numbers (Encoding)
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})
df['c_charge_degree'] = df['c_charge_degree'].map({'F': 1, 'M': 0}) # Felony=1, Misdemeanor=0

# One-hot encode 'race' (creates columns like race_African-American, race_Caucasian, etc.)
df = pd.get_dummies(df, columns=['race'], drop_first=False)

# 4. Split into Training and Testing sets
X = df.drop(target, axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Data loaded and processed!")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print("\nFirst 5 rows of training data:")
X_train.head()

---
# **Step 3: Train Your Model**

You can use the example below (Logistic Regression) or replace it with any model you like (Random Forest, Neural Networks, XGBoost, etc.).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Initialize the model
# Feel free to replace this with any other sklearn compatible model!
model = LogisticRegression(max_iter=1000)

# 2. Train the model
model.fit(X_train, y_train)

# 3. Generate predictions on the test set
predictions = model.predict(X_test)

# 4. Evaluate accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"✅ Model Trained! Accuracy: {accuracy:.2%}")

---
# **Step 4: Connect to the Leaderboard**

This step connects your notebook to the specific backend for the Justice & Equity Challenge. 

*Note: You will be prompted to enter a username and password. If you don't have one, check with your instructor or the challenge website.*

In [ ]:
from aimodelshare.aws import set_credentials
from aimodelshare.playground import Experiment

# The specific Model Playground URL for the Justice Challenge
my_playground_url = "https://cf3wdpkg0d.execute-api.us-east-1.amazonaws.com/prod/m"

# Set your credentials (pop-up will appear)
set_credentials(apiurl=my_playground_url)

# Connect to the experiment
playground = Experiment(my_playground_url)

---
# **Step 5: Submit & Check Results**

Submit your model to the leaderboard. You can include a description and tags to help you remember which model this was.

In [ ]:
# 1. Submit your model
playground.submit_model(
    model=model,
    prediction_submission=predictions,
    input_dict={
        "description": "Logistic Regression Baseline", 
        "tags": "sklearn, logistic_regression, advanced_pathway"
    }
)

print("✅ Model submitted successfully!")

# 2. Check the leaderboard
print("Loading leaderboard...")
leaderboard = playground.get_leaderboard()
playground.stylize_leaderboard(leaderboard)